# 03 - Two-pass labeling

**Question this notebook answers:** which crops are arched, according to a human?

The pretrained detector gave you cows. Nothing has given you posture. That label
does not exist in any public dataset at the granularity this project needs, so it
is created here. Posture and geometry are deliberately collected in separate
passes so a computed curve cannot steer the human posture decision.

## Non-negotiable rules, enforced in code

1. **The split is locked before labeling.** Otherwise the boundary can drift
   toward whatever makes the numbers look good.
2. **Pass A is image-only.** It shows no keypoint, sagitta, model probability,
   source score, or existing geometry.
3. **Test and validation are labeled blind**, in random order. Active ordering
   is a train-only operation and the seed model fits train labels only.
4. **A model may reorder the queue; only you write a label.** There is no code
   path from a prediction into the `label` column.
5. **Pass B never rewrites posture.** It admits only completed `arched`/`normal`
   rows and records geometry under a separate reviewer field.
6. **No thresholding geometry into labels.** Sagitta is a feature, never ground truth.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_colwidth", 60)
print("project root:", PROJECT_ROOT)

## Interactive backend

Pass B point capture needs click events, which need the widget backend. If its
figure is static, install/enable `ipympl` and restart the kernel. Pass A posture
buttons do not require click coordinates.

In [ ]:
%matplotlib widget
import ipywidgets
print("ipywidgets", ipywidgets.__version__)
print("matplotlib backend:", plt.get_backend() if "plt" in dir() else "not imported yet")

In [ ]:
MAX_SAMPLES = None

MANIFEST_CSV = PROJECT_ROOT / "data" / "manifest.csv"
POSTURE_REVIEWER = "anil-posture"
GEOMETRY_REVIEWER = "anil-geometry"

## 0. Split lock check

`scripts/03_split.py` must have run. It splits by `video_id` group, so no frame
of a given video can appear in two splits.

In [ ]:
from cowarch.io import as_bool, read_manifest
from cowarch.labeling import label_summary
from cowarch.splits import assert_no_group_leakage

manifest = read_manifest(MANIFEST_CSV)
accepted = manifest.loc[as_bool(manifest["accepted"])]
if accepted["split"].astype(str).str.strip().eq("").any():
    raise RuntimeError(
        "Some accepted rows have no split. Lock the split first:\n"
        "  python scripts/03_split.py --manifest data/manifest.csv --group-column video_id"
    )
assert_no_group_leakage(accepted, "video_id")
print("split locked, no group leakage")
display(accepted.groupby("split").agg(crops=("sample_id", "size"), groups=("video_id", "nunique")))
display(label_summary(MANIFEST_CSV))

## Pass A - posture label definitions

Read these before every session; drift between sessions is a real annotation
failure mode.

| Label | Definition |
|---|---|
| `arched` | Clear dorsal convexity over the thoracolumbar region relative to the withers-sacrum line. |
| `normal` | Straight or near-straight dorsal line. |
| `uncertain` | The image is usable but you cannot decide confidently. |
| `invalid` | Not a lateral view, heavy occlusion, body cut off, animal turning or lying down. |

Use `uncertain` / `invalid` freely:

- **head down** (grazing, drinking) - the dorsal line is altered by posture, not necessarily by pain
- the apparent curve comes only from the **head-and-tail** outline, not the spine
- the animal is **turning** and the silhouette is a rotated projection

`uncertain` and `invalid` are excluded from training, but their **counts are
reported**. A high `uncertain` rate is a finding about the data, not a failure.

`PostureLabeler` displays the crop without points, overlays, identifiers, model
probabilities, source scores, or geometry values. Geometry comes later in Pass B.

## 1. Test split first, blind

Labeling test first, before any model exists, removes the possibility of the
model influencing the ground truth it will be judged against.

In [ ]:
from cowarch.labeling import PostureLabeler

test_labeler = PostureLabeler(
    MANIFEST_CSV,
    split="test",
    reviewer=POSTURE_REVIEWER,
    order="random",           # blind: enforced for non-train splits
)
test_labeler.display()

## 2. Validation split, also blind

In [ ]:
val_labeler = PostureLabeler(
    MANIFEST_CSV,
    split="val",
    reviewer=POSTURE_REVIEWER,
    order="random",
)
val_labeler.display()

## 3. Training seed: 5 obvious arched + 5 obvious normal

Pick unambiguous examples. This is a *prototype seed*, not a training set - ten
labels cannot support a performance claim, only a ranking that makes the next
round of labeling cheaper.

In [ ]:
seed_labeler = PostureLabeler(
    MANIFEST_CSV,
    split="train",
    reviewer=POSTURE_REVIEWER,
    order="random",
    max_samples=30,           # scan up to 30 to find ~5 clear cases per class
)
seed_labeler.display()

In [ ]:
summary = label_summary(MANIFEST_CSV)
display(summary)
train_counts = summary.loc["train"] if "train" in summary.index else None
if train_counts is not None:
    ready = train_counts.get("arched", 0) >= 3 and train_counts.get("normal", 0) >= 3
    print(f"seed ready for active learning: {ready} "
          f"(arched={train_counts.get('arched', 0)}, normal={train_counts.get('normal', 0)})")

## 4. Frozen embeddings

ImageNet ResNet18 with the classifier head removed, weights never updated. On a
few hundred crops this is a fixed feature extractor, not a trained model - which
is the point: with ten labels there is nothing to train.

In [ ]:
from cowarch.embeddings import extract_resnet18_embeddings
from cowarch.io import resolve_data_path

EMBEDDING_CACHE = PROJECT_ROOT / "outputs" / "embeddings_all.npz"

manifest = read_manifest(MANIFEST_CSV)
accepted_positions = np.flatnonzero(as_bool(manifest["accepted"]).to_numpy())
pool = manifest.iloc[accepted_positions].reset_index(drop=True)
sample_ids = pool["sample_id"].astype(str).to_numpy()

if EMBEDDING_CACHE.exists():
    cached = np.load(EMBEDDING_CACHE, allow_pickle=False)
    if np.array_equal(cached["sample_ids"].astype(str), sample_ids):
        embeddings = cached["embeddings"]
        print(f"cache hit: {embeddings.shape}")
    else:
        embeddings = None
else:
    embeddings = None

if embeddings is None:
    paths = [resolve_data_path(p, MANIFEST_CSV) for p in pool["crop_path"].astype(str)]
    embeddings = extract_resnet18_embeddings(paths)
    EMBEDDING_CACHE.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(EMBEDDING_CACHE, sample_ids=sample_ids, embeddings=embeddings)
    print(f"computed and cached: {embeddings.shape}")

## 5. Active learning round

The seed model can score every accepted crop and hands back the **train** rows it is least
sure about - the frames near its decision boundary, where your label carries the
most information. Labeling 20 uncertain frames beats labeling 20 random ones,
and beats scanning hundreds blind.

Under ~8 train labels per class it uses cosine nearest-centroid; above that it
switches to logistic regression. Fitting uses accepted `arched`/`normal` train
rows only. Queue selection uses accepted, unlabeled train rows only. Explicitly
requesting validation or test in `labelable_splits` raises instead of silently
broadening the pool. Returned probabilities still cover all rows for diagnostics,
but this notebook never passes them into the posture UI.

In [ ]:
from cowarch.active import select_next_batch

BATCH_SIZE = 20

# Reload decisions written by the previous labeler round. Embeddings stay
# valid only while the accepted sample order is unchanged.
manifest = read_manifest(MANIFEST_CSV)
accepted_positions = np.flatnonzero(as_bool(manifest["accepted"]).to_numpy())
pool = manifest.iloc[accepted_positions].reset_index(drop=True)
fresh_sample_ids = pool["sample_id"].astype(str).to_numpy()
if not np.array_equal(fresh_sample_ids, sample_ids):
    raise RuntimeError(
        "Accepted samples changed since embeddings were loaded. Rerun section 4 "
        "before selecting another active-learning batch."
    )

batch, model, probabilities = select_next_batch(pool, embeddings, batch_size=BATCH_SIZE)
print(f"seed model: {model.kind} trained on {model.n_labeled} labels")
print(f"selected {len(batch)} frames closest to the decision boundary")

priority = pd.Series(-np.inf, index=manifest.index, dtype=float)
manifest_batch = accepted_positions[batch]
priority.iloc[manifest_batch] = np.linspace(1.0, 0.01, len(batch))
assert pool.iloc[batch]["split"].eq("train").all()

### Verify the queue boundary

This check exposes only aggregate split membership. Per-row model probabilities
and source scores stay out of Pass A.

In [ ]:
queue_rows = pool.iloc[batch]
assert queue_rows["split"].eq("train").all()
print(queue_rows["split"].value_counts().to_dict())

### Label the uncertain batch

`order="priority"` is accepted here only because the split is `train`. The same
call against `val` or `test` raises.

In [ ]:
active_labeler = PostureLabeler(
    MANIFEST_CSV,
    split="train",
    reviewer=POSTURE_REVIEWER,
    order="priority",
    priority=priority,
    max_samples=len(batch),
)
active_labeler.display()

## 6. Repeat

Re-run section 5 (batch selection) and the train labeler above. Each round the model is
fit on more labels, so the frames it asks about get more informative. Stop when
the queue stops surprising you, or when you hit the target below.

Rough targets for this PoC:

| Split | Target labeled crops |
|---|---|
| train | 60-100 |
| val | 20-30 |
| test | 30-40 |

Validation and test frames are never fit or queued by active learning. Their counts
grow only through the blind random labelers in sections 1 and 2.

In [ ]:
summary = label_summary(MANIFEST_CSV)
display(summary)

labeled = summary[["arched", "normal"]].sum(axis=1) if len(summary) else pd.Series(dtype=int)
targets = {"train": 60, "val": 20, "test": 30}
for split, target in targets.items():
    have = int(labeled.get(split, 0))
    bar = "#" * int(20 * min(have / target, 1.0))
    print(f"{split:6s} {have:4d}/{target:<4d} |{bar:<20s}|")

if len(summary):
    unusable = summary[["uncertain", "invalid"]].sum().sum()
    total = summary.drop(columns=["unlabeled"], errors="ignore").sum().sum()
    if total:
        print(f"\nuncertain+invalid: {unusable}/{total} = {unusable / total:.1%} - report this figure")

## 7. Pass B - dorsal geometry

Run this only after Pass A. `DorsalGeometryLabeler` queues accepted rows whose
posture is already `arched` or `normal`; `uncertain`, `invalid`, and unlabeled
rows are ineligible. It intentionally hides the saved posture and has no posture
buttons, so saving geometry cannot rewrite `label`.

Choose one protocol before a labeling campaign:

- `mode="keypoints"`: click `withers -> thoracic -> thoracolumbar -> lumbar -> sacrum`.
- `mode="anchors"`: click only `withers -> sacrum`; these anchors support the
  dense-contour baseline when a mask is available.

No live sagitta or geometry score is shown while points are being placed. For
five-point mode, a feature summary becomes available only after **save geometry**
has persisted the points.

In [ ]:
from cowarch.labeling import DorsalGeometryLabeler

GEOMETRY_SPLIT = "train"       # rerun explicitly for val/test if geometry is required there
GEOMETRY_MODE = "keypoints"    # or "anchors"; keep one protocol per campaign

geometry_labeler = DorsalGeometryLabeler(
    MANIFEST_CSV,
    split=GEOMETRY_SPLIT,
    reviewer=GEOMETRY_REVIEWER,
    mode=GEOMETRY_MODE,
    order="random",
    max_samples=MAX_SAMPLES,
)
geometry_labeler.display()

### Audit fields and legacy manifests

New decisions use `posture_reviewed_by`, `geometry_reviewed_by`, and
`annotation_pass` (`posture` or `geometry`). The legacy `reviewed_by` column is
retained: new Pass A decisions still populate it, and an old non-empty value is
interpreted as the posture reviewer. If an old row already contains geometry,
that same legacy reviewer is also migrated in memory as its geometry reviewer.
The added columns are persisted on the next annotation save.

## What this labeling does and does not license

Done:
- a posture label that no public dataset provided
- blind test and validation annotation
- separate posture and geometry audit fields, with legacy `reviewed_by` support

Still open, and belonging in the report's limitations:
- **one annotator.** No inter-rater agreement, so the "ground truth" is one
  person's reading of a visual definition.
- **active learning shapes the training distribution.** The train split is
  deliberately enriched near the boundary and no longer reflects source
  prevalence. Test, sampled blind, is the honest measurement.
- **educational footage over-represents lame animals.** Neither split reflects
  herd prevalence.

Next: **04_training_evaluation.ipynb**.